# Notebook 2 — Combined GEO Methods (Parallel)

Applies all 11 combined GEO methods to a defined range of retail target documents.
Each notebook instance works on its own JSON file — no write conflicts.

## Parallel Setup
| Notebook | File | Range |
|---|---|---|
| A | `selected_docs_A.json` | 0 – 124 |
| B | `selected_docs_B.json` | 125 – 249 |
| C | `selected_docs_C.json` | 250 – 374 |
| D | `selected_docs_D.json` | 375 – 499 |

## Instructions
1. Run **Setup**
2. Set `NOTEBOOK_ID`, `QUERY_START`, `QUERY_END` in **Parameters**
3. Run **Create Subset File** — creates `selected_docs_{ID}.json`
4. Run **Manipulation**
5. After all 4 notebooks finish: run **Merge Results**
6. Run **Summary**

## Setup

In [1]:
import json
import os
import sys
import time
from datetime import datetime

notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))
sys.path.insert(0, os.path.join(project_root, "src"))

from llms import OpenAIHelper
from methods import ALL_COMBINED_METHODS

config_path = os.path.join(project_root, "config.json")
with open(config_path, "r") as f:
    config = json.load(f)
os.environ["OPENAI_API_KEY"] = config["OPENAI_API_KEY"]

data_dir = os.path.join(project_root, "data", "retail")
source_path = os.path.join(data_dir, "selected_docs.json")

print("Setup complete.")
print(f"Project root: {project_root}")
print(f"Methods: {list(ALL_COMBINED_METHODS.keys())}")

Setup complete.
Project root: /Users/leonardrampf/Library/CloudStorage/OneDrive-Personal/Dokumente/Universität/Nova SBE/Work Project/geo-experiment
Methods: ['FC', 'FQ', 'FS', 'CQ', 'CS', 'QS', 'FCQ', 'FCS', 'FQS', 'CQS', 'FCQS']


## Parameters

**Change these values for each notebook instance.**

In [2]:
# LLM
LLM_NAME = "gpt-4o-mini-2024-07-18"

# <- SET THESE FOR EACH NOTEBOOK
NOTEBOOK_ID = "D"   # A, B, C, or D
QUERY_START = 375     # inclusive
QUERY_END   = 499   # inclusive

# Output file for this notebook
subset_path = os.path.join(data_dir, f"selected_docs_{NOTEBOOK_ID}.json")

# Methods
METHOD_NAMES = list(ALL_COMBINED_METHODS.keys())

print(f"Notebook ID:     {NOTEBOOK_ID}")
print(f"Query range:     {QUERY_START} – {QUERY_END}")
print(f"Output file:     selected_docs_{NOTEBOOK_ID}.json")
print(f"LLM:             {LLM_NAME}")
print(f"Methods:         {METHOD_NAMES}")

Notebook ID:     D
Query range:     375 – 499
Output file:     selected_docs_D.json
LLM:             gpt-4o-mini-2024-07-18
Methods:         ['FC', 'FQ', 'FS', 'CQ', 'CS', 'QS', 'FCQ', 'FCS', 'FQS', 'CQS', 'FCQS']


## Create Subset File

Creates `selected_docs_{ID}.json` with only the queries in the defined range.
Skipped if the file already exists.

In [3]:
if os.path.exists(subset_path):
    print(f"selected_docs_{NOTEBOOK_ID}.json already exists — skipping creation.")
else:
    with open(source_path, "r", encoding="utf-8") as f:
        all_docs = json.load(f)

    subset = {
        idx: all_docs[idx]
        for idx in all_docs
        if QUERY_START <= int(idx) <= QUERY_END
    }

    with open(subset_path, "w", encoding="utf-8") as f:
        json.dump(subset, f, indent=4, ensure_ascii=False)

    print(f"Created selected_docs_{NOTEBOOK_ID}.json with {len(subset)} queries ({QUERY_START}–{QUERY_END})")

Created selected_docs_D.json with 125 queries (375–499)


## Manipulation

Reads and writes only `selected_docs_{ID}.json` — no conflicts with other notebooks.
Safe to interrupt and resume.

In [ ]:
llm = OpenAIHelper(LLM_NAME)

with open(subset_path, "r", encoding="utf-8") as f:
    selected_docs = json.load(f)

query_indices = sorted(selected_docs.keys(), key=lambda x: int(x))
total = len(query_indices)

start_time = datetime.now()
print(f"Started at: {start_time.strftime('%H:%M:%S')}")
print(f"Notebook {NOTEBOOK_ID}: queries {QUERY_START}–{QUERY_END} ({total} docs)")
print()

for method_name in METHOD_NAMES:
    key = f"{method_name}(doc)"

    already_done = sum(
        1 for idx in query_indices
        for entry in selected_docs[idx].values()
        if entry.get(key) is not None
    )

    if already_done == total:
        print(f"[{method_name}] All {total} docs already done — skipping")
        continue

    method = ALL_COMBINED_METHODS[method_name](llm)
    print(f"[{method_name}] Starting ({already_done}/{total} already done)")

    for i, query_idx in enumerate(query_indices):
        for doc_idx, entry in selected_docs[query_idx].items():

            if entry.get(key) is not None:
                continue

            doc = entry["doc"]

            try:
                result = method.improve_text(doc)
                selected_docs[query_idx][doc_idx][key] = result
                with open(subset_path, "w", encoding="utf-8") as f:
                    json.dump(selected_docs, f, indent=4, ensure_ascii=False)
                print(f"  [{i+1}/{total}] query {query_idx}: done")

            except Exception as e:
                print(f"  [{i+1}/{total}] query {query_idx}: ERROR — {e}")
                time.sleep(10)
                try:
                    result = method.improve_text(doc)
                    selected_docs[query_idx][doc_idx][key] = result
                    with open(subset_path, "w", encoding="utf-8") as f:
                        json.dump(selected_docs, f, indent=4, ensure_ascii=False)
                    print(f"  [{i+1}/{total}] query {query_idx}: done (retry OK)")
                except Exception as e2:
                    print(f"  [{i+1}/{total}] query {query_idx}: FAILED — {e2}")
                    selected_docs[query_idx][doc_idx][key] = None

    done = sum(
        1 for idx in query_indices
        for entry in selected_docs[idx].values()
        if entry.get(key) is not None
    )
    print(f"[{method_name}] Complete: {done}/{total}")
    print()

end_time = datetime.now()
elapsed = end_time - start_time
print(f"{'='*60}")
print(f"NOTEBOOK {NOTEBOOK_ID} COMPLETE")
print(f"Started:    {start_time.strftime('%H:%M:%S')}")
print(f"Finished:   {end_time.strftime('%H:%M:%S')}")
print(f"Total time: {str(elapsed).split('.')[0]}")

Started at: 08:45:30
Notebook D: queries 375–499 (125 docs)

[FC] All 125 docs already done — skipping
[FQ] All 125 docs already done — skipping
[FS] All 125 docs already done — skipping
[CQ] All 125 docs already done — skipping
[CS] All 125 docs already done — skipping
[QS] All 125 docs already done — skipping
[FCQ] Starting (124/125 already done)
  [125/125] query 499: done
[FCQ] Complete: 125/125

[FCS] Starting (0/125 already done)
  [1/125] query 375: done
  [2/125] query 376: done
  [3/125] query 377: done
  [4/125] query 378: done
  [5/125] query 379: done
  [6/125] query 380: done
  [7/125] query 381: done
  [8/125] query 382: done
  [9/125] query 383: done
  [10/125] query 384: done
  [11/125] query 385: done
  [12/125] query 386: done
  [13/125] query 387: done
  [14/125] query 388: done
  [15/125] query 389: done
  [16/125] query 390: done
  [17/125] query 391: done
  [18/125] query 392: done
  [19/125] query 393: done
  [20/125] query 394: done
  [21/125] query 395: done
  

## Merge Results

Run this cell **only after all 4 notebooks are complete**.
Merges A, B, C, D back into `selected_docs.json`.

In [ ]:
NOTEBOOK_IDS = ["A", "B", "C", "D"]

merged = {}
for nb_id in NOTEBOOK_IDS:
    path = os.path.join(data_dir, f"selected_docs_{nb_id}.json")
    if not os.path.exists(path):
        print(f"WARNING: selected_docs_{nb_id}.json not found — skipping.")
        continue
    with open(path, "r", encoding="utf-8") as f:
        subset = json.load(f)
    merged.update(subset)
    print(f"Loaded selected_docs_{nb_id}.json ({len(subset)} queries)")

# Sort by query index
merged = dict(sorted(merged.items(), key=lambda x: int(x[0])))

output_path = os.path.join(data_dir, "selected_docs.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(merged, f, indent=4, ensure_ascii=False)

print(f"\nMerged {len(merged)} queries into selected_docs.json")

## Summary — Completion Status

In [ ]:
with open(os.path.join(data_dir, "selected_docs.json"), "r") as f:
    all_docs = json.load(f)

total = len(all_docs)
print(f"RETAIL — full dataset ({total} docs)")
print()
print("Combined methods:")
for method_name in METHOD_NAMES:
    key = f"{method_name}(doc)"
    done = sum(1 for q in all_docs.values() for e in q.values() if e.get(key) is not None)
    bar = "\u2588" * (done * 20 // total) + "\u2591" * (20 - done * 20 // total)
    print(f"  {key:25} [{bar}] {done}/{total}")